In [1]:
!pip install requests pandas rasterio

In [2]:
import requests
import pandas as pd
import rasterio
from io import BytesIO

# Define Sri Lankan locations

In [3]:
locations = {
    "Kandy": (7.2906, 80.6337),
    "Matale": (7.4675, 80.6234),
    "Nuwara Eliya": (6.9497, 80.7891),
    "Badulla": (6.9934, 81.0550),
    "Kurunegala": (7.4863, 80.3647),
    "Anuradhapura": (8.3114, 80.4037),
    "Polonnaruwa": (7.9403, 81.0188),
    "Ampara": (7.2912, 81.6724),
    "Hambantota": (6.1429, 81.1212),
    "Monaragala": (6.8728, 81.3507)
}

print(locations)

{'Kandy': (7.2906, 80.6337), 'Matale': (7.4675, 80.6234), 'Nuwara Eliya': (6.9497, 80.7891), 'Badulla': (6.9934, 81.055), 'Kurunegala': (7.4863, 80.3647), 'Anuradhapura': (8.3114, 80.4037), 'Polonnaruwa': (7.9403, 81.0188), 'Ampara': (7.2912, 81.6724), 'Hambantota': (6.1429, 81.1212), 'Monaragala': (6.8728, 81.3507)}


# Define the SoilGrids properties

In [4]:
properties = {
    "ph": "phh2o",
    "nitrogen": "nitrogen",
    "clay": "clay",
    "sand": "sand",
    "silt": "silt",
    "soc": "soc"
}

print(properties)

{'ph': 'phh2o', 'nitrogen': 'nitrogen', 'clay': 'clay', 'sand': 'sand', 'silt': 'silt', 'soc': 'soc'}


# Create the SoilGrids WCS function

In [5]:
def get_soil_value(lat, lon, soil_property):

    url = "https://maps.isric.org/mapserv"

    map_file = f"/map/{soil_property}.map"

    params = {
        "map": map_file,
        "SERVICE": "WCS",
        "VERSION": "2.0.1",
        "REQUEST": "GetCoverage",
        "COVERAGEID": f"{soil_property}_0-5cm_Q0.5",
        "FORMAT": "image/tiff",
        "SUBSET": [
            f"long({lon-0.01},{lon+0.01})",
            f"lat({lat-0.01},{lat+0.01})"
        ],
        "SUBSETTINGCRS": "http://www.opengis.net/def/crs/EPSG/0/4326",
        "OUTPUTCRS": "http://www.opengis.net/def/crs/EPSG/0/4326"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code)
        return None

    try:
        with rasterio.open(BytesIO(response.content)) as dataset:
            data = dataset.read(1)

            # Remove NoData values
            nodata = dataset.nodata

            if nodata is not None:
                data = data[data != nodata]

            if len(data) == 0:
                return None

            return float(data.mean())

    except Exception as e:
        print("Reading error:", e)
        return None

# Test ONE location first

In [6]:
lat, lon = locations["Kandy"]

ph_value = get_soil_value(lat, lon, "phh2o")

print("Kandy Soil pH:", ph_value)

Kandy Soil pH: 15.444444444444445


# Extract all properties

In [7]:
data = []

for district, (lat, lon) in locations.items():

    print("Processing:", district)

    row = {
        "district": district,
        "latitude": lat,
        "longitude": lon
    }

    for name, soil_property in properties.items():

        value = get_soil_value(
            lat,
            lon,
            soil_property
        )

        row[name] = value

    data.append(row)

soil_df = pd.DataFrame(data)

soil_df

Processing: Kandy
Processing: Matale
Processing: Nuwara Eliya
Processing: Badulla
Processing: Kurunegala
Processing: Anuradhapura
Processing: Polonnaruwa
Processing: Ampara
Processing: Hambantota
Processing: Monaragala


,district,latitude,longitude,ph,nitrogen,clay,sand,silt,soc
0,Kandy,7.2906,80.6337,15.444444,85.013889,95.875000,106.500000,58.458333,102.472222
1,Matale,7.4675,80.6234,20.138889,108.805556,85.708333,157.861111,80.458333,109.652778
2,Nuwara Eliya,6.9497,80.7891,19.513889,145.875000,121.097222,128.319444,81.319444,178.736111
3,Badulla,6.9934,81.0550,20.805556,128.736111,114.250000,143.736111,84.638889,114.986111
4,Kurunegala,7.4863,80.3647,14.500000,86.958333,92.486111,85.000000,67.513889,84.375000
5,Anuradhapura,8.3114,80.4037,54.861111,177.180556,194.611111,480.375000,157.194444,173.041667
6,Polonnaruwa,7.9403,81.0188,56.958333,306.708333,196.291667,255.694444,303.083333,260.319444
7,Ampara,7.2912,81.6724,24.458333,123.333333,120.083333,130.486111,95.194444,136.194444
8,Hambantota,6.1429,81.1212,49.597222,145.722222,187.319444,273.208333,177.513889,163.833333
9,Monaragala,6.8728,81.3507,53.597222,237.569444,243.708333,370.611111,204.819444,260.347222


# Convert SoilGrids values

In [8]:
soil_df["ph"] = soil_df["ph"] / 10
soil_df["nitrogen"] = soil_df["nitrogen"] / 100
soil_df["clay"] = soil_df["clay"] / 10
soil_df["sand"] = soil_df["sand"] / 10
soil_df["silt"] = soil_df["silt"] / 10
soil_df["soc"] = soil_df["soc"] / 10

In [9]:
soil_df

,district,latitude,longitude,ph,nitrogen,clay,sand,silt,soc
0,Kandy,7.2906,80.6337,1.544444,0.850139,9.587500,10.650000,5.845833,10.247222
1,Matale,7.4675,80.6234,2.013889,1.088056,8.570833,15.786111,8.045833,10.965278
2,Nuwara Eliya,6.9497,80.7891,1.951389,1.458750,12.109722,12.831944,8.131944,17.873611
3,Badulla,6.9934,81.0550,2.080556,1.287361,11.425000,14.373611,8.463889,11.498611
4,Kurunegala,7.4863,80.3647,1.450000,0.869583,9.248611,8.500000,6.751389,8.437500
5,Anuradhapura,8.3114,80.4037,5.486111,1.771806,19.461111,48.037500,15.719444,17.304167
6,Polonnaruwa,7.9403,81.0188,5.695833,3.067083,19.629167,25.569444,30.308333,26.031944
7,Ampara,7.2912,81.6724,2.445833,1.233333,12.008333,13.048611,9.519444,13.619444
8,Hambantota,6.1429,81.1212,4.959722,1.457222,18.731944,27.320833,17.751389,16.383333
9,Monaragala,6.8728,81.3507,5.359722,2.375694,24.370833,37.061111,20.481944,26.034722


# Save as CSV

In [10]:
soil_df.to_csv(
    "sri_lanka_soil.csv",
    index=False
)

print("CSV created successfully!")

CSV created successfully!


In [11]:
from google.colab import files

files.download("sri_lanka_soil.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>